### Chatbot And RAG Evaluation

Retrieval Augmented Generation (RAG) is a technique that enhances Large Language Models (LLMs) by providing them with relevant external knowledge. It has become one of the most widely used approaches for building LLM applications.

This tutorial will show you how to evaluate your RAG applications using LangSmith. You'll learn:

1. How to create test datasets
2. How to run your RAG application on those datasets
3. How to measure your application's performance using different evaluation metrics

#### Overview
A typical RAG evaluation workflow consists of three main steps:

1. Creating a dataset with questions and their expected answers
2. Running your RAG application on those questions
3. Using evaluators to measure how well your application performed, looking at factors like:
 - Answer relevance
 - Answer accuracy
 - Retrieval quality
 
For this tutorial, we'll create and evaluate a bot that answers questions about a few of Lilian Weng's insightful blog posts.

### Chatbot Evaluation

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGSMITH_API_KEY"]=os.getenv("LANGSMITH_API_KEY")
os.environ["HUGGINGFACEHUB_API_TOKEN"] = os.getenv("HUGGINGFACEHUB_API_TOKEN")
os.environ["LANGSMITH_TRACING"]="true"

In [10]:
from langsmith import Client

client = Client()

# Define dataset: these are your test cases
dataset_name = "Chatbots Evaluation"
dataset = client.create_dataset(dataset_name)
client.create_examples(
    dataset_id=dataset.id,
    examples=[
        {
            "inputs": {"question": "What is LangChain?"},
            "outputs": {"answer": "A framework for building LLM applications"},
        },
        {
            "inputs": {"question": "What is LangSmith?"},
            "outputs": {"answer": "A platform for observing and evaluating LLM applications"},
        },
        {
            "inputs": {"question": "What is OpenAI?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        },
        {
            "inputs": {"question": "What is Google?"},
            "outputs": {"answer": "A technology company known for search"},
        },
        {
            "inputs": {"question": "What is Mistral?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        }
    ]
)

{'example_ids': ['9c89c3bd-a334-4405-9829-70ca3fc42fb9',
  'df1e6dd7-e767-4f28-938c-24413d43205d',
  '980037c3-a43b-454d-8611-d559f657f93e',
  '46ad5c50-3144-4869-8d90-7544bc39ed3a',
  '75c8c24e-468b-4347-bc4e-f369d752a282'],
 'count': 5,
 'as_of': '2026-07-02T04:54:51.397758334Z'}

### Define Metrics (LLM As A Judge)


In [4]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
import os


llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    huggingfacehub_api_token=os.environ["HUGGINGFACEHUB_API_TOKEN"],
)

judge_llm = ChatHuggingFace(llm=llm)

eval_instructions = "You are an expert professor specialized in grading students' answers to questions."

def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    user_content = f"""You are grading the following question:
    {inputs['question']}
    Here is the real answer:
    {reference_outputs['answer']}
    You are grading the following predicted answer:
    {outputs['response']}
    Respond with CORRECT or INCORRECT:
    Grade:
    """
    response = judge_llm.invoke([
        {"role": "system", "content": eval_instructions},
        {"role": "user", "content": user_content}
    ]).content.strip()
    return "CORRECT" in response

c:\Users\sushm\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
## Concisions- checks whether the actual output is less than 2x the length of the expected result.

def concision(outputs: dict, reference_outputs: dict) -> bool:
    return int(len(outputs["response"]) < 2 * len(reference_outputs["answer"]))

### Run Evaluations

In [6]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
import os

default_instructions = "Respond to the users question in a short, concise manner (one short sentence)."

def my_app(question: str, model: str = "mistralai/Mistral-7B-Instruct-v0.3", instructions: str = default_instructions) -> str:
    endpoint = HuggingFaceEndpoint(
        repo_id=model, 
        temperature=0.1, 
        huggingfacehub_api_token=os.getenv('HUGGINGFACEHUB_API_TOKEN')
    )
    llm = ChatHuggingFace(llm=endpoint)
    return llm.invoke([
        {"role": "system", "content": instructions},
        {"role": "user", "content": question},
    ]).content

In [7]:
### Call my_app for every datapoints
def ls_target(inputs: str) -> dict:
    return {"response": my_app(inputs["question"])}

In [11]:


def ls_target(inputs: dict) -> dict:
    response = llm.invoke(inputs["question"])
    return {"response": response.content}


# Run evaluation
experiment_results = client.evaluate(
    ls_target,
    data=dataset_name,
    evaluators=[correctness, concision],
    experiment_prefix="llama-chatbot",
    max_concurrency=1,
)

View the evaluation results for experiment: 'llama-chatbot-85bc2965' at:
https://smith.langchain.com/o/53a7b2e9-c45e-46c3-9695-0fc593dd50f0/datasets/ba4a957d-2d10-44b7-906a-4984edbe57b1/compare?selectedSessions=7c87fc74-813f-47a3-bb65-917cded4e6c3




5it [00:15,  3.05s/it]


In [12]:
### Call my_app for every datapoints
def ls_target(inputs: dict) -> dict:
    return {
        "response": my_app(
            inputs["question"],
            model="meta-llama/Llama-3.1-8B-Instruct"
        )
    }

In [13]:
## Run our evaluation
experiment_results = client.evaluate(
    ls_target,  # Your AI system
    data=dataset_name,
    evaluators=[correctness, concision],
    experiment_prefix="llama-chatbot",
    max_concurrency=1,  # Recommended for HF Router
)

View the evaluation results for experiment: 'llama-chatbot-da817be8' at:
https://smith.langchain.com/o/53a7b2e9-c45e-46c3-9695-0fc593dd50f0/datasets/ba4a957d-2d10-44b7-906a-4984edbe57b1/compare?selectedSessions=672a4953-5a5b-4dd3-a741-a6f9a3be0013




5it [00:07,  1.54s/it]


### Evaluation For RAG

In [17]:
## RAG
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# List of URLs to load documents from
urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/",
]

# Load documents
docs = [WebBaseLoader(url).load() for url in urls]
docs_list = [item for sublist in docs for item in sublist]

# Split into chunks
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=250,
    chunk_overlap=0,
)
doc_splits = text_splitter.split_documents(docs_list)

# Create vector store
vectorstore = InMemoryVectorStore.from_documents(
    documents=doc_splits,
    embedding=HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    ),
)

# Retriever
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 6}
)

c:\Users\sushm\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sushm\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2191.92it/s]


In [15]:
!pip install sentence-transformers

^C


   ---------------------------------------- 0.0/596.4 kB ? eta -:--:--
   ---------------------------------------- 0.0/596.4 kB ? eta -:--:--
   ---------------------------------------- 0.0/596.4 kB ? eta -:--:--
   ---------------------------------------- 0.0/596.4 kB ? eta -:--:--
   ----------------- ---------------------- 262.1/596.4 kB ? eta -:--:--
   ----------------- ---------------------- 262.1/596.4 kB ? eta -:--:--
   --------------------------------- ---- 524.3/596.4 kB 479.2 kB/s eta 0:00:01
   ---------------------------------------- 596.4/596.4 kB 509.6 kB/s  0:00:01
   ---------------------------------------- 0.0/11.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.2 MB ? eta -:--:--
    --------------------------------------- 0.3/11.2 MB ? eta -:--:--
    --------------------------------------- 0.3/11.2 MB ? eta -:--:--
   - -------------------------------------- 0.5/11.2 MB 508.0

ERROR: Could not install packages due to an OSError: [WinError 32] The process cannot access the file because it is being used by another process: 'c:\\users\\sushm\\appdata\\local\\programs\\python\\python312\\lib\\site-packages\\setuptools\\_vendor\\packaging\\_manylinux.py'
Consider using the `--user` option or check the permissions.



In [18]:
retriever.invoke("what is agents")

[Document(id='89f9d06a-304b-4215-a610-e4c1cdc80a1d', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and refine them for future steps, 

In [19]:
from langchain_openai import ChatOpenAI
import os

llm = ChatOpenAI(
    model="meta-llama/Llama-3.1-8B-Instruct",
    base_url="https://router.huggingface.co/v1",
    api_key=os.environ["HUGGINGFACEHUB_API_TOKEN"],
    temperature=0,
)

In [20]:
from langsmith import traceable

## Add decorator
@traceable()
def rag_bot(question:str)->dict:
    ## Relevant context
    docs=retriever.invoke(question)
    docs_string = " ".join(doc.page_content for doc in docs)

    instructions = f"""You are a helpful assistant who is good at analyzing source information and answering questions.       Use the following source documents to answer the user's questions.       If you don't know the answer, just say that you don't know.       Use three sentences maximum and keep the answer concise.

Documents:
{docs_string}"""
    
    ## llm invoke

    ai_msg=llm.invoke([
        {"role": "system", "content": instructions},
        {"role": "user", "content": question},

    ])
    return {"answer":ai_msg.content,"documents":docs}



In [21]:
rag_bot("What is agents")

{'answer': 'In the context of the provided source documents, an agent refers to a software program or a system that can perform tasks, make decisions, and interact with its environment, often using a large language model (LLM) as its core controller.',
 'documents': [Document(id='89f9d06a-304b-4215-a610-e4c1cdc80a1d', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: T

### Dataset

In [22]:
from langsmith import Client

client=Client()

# Define the examples for the dataset
examples = [
    {
        "inputs": {"question": "How does the ReAct agent use self-reflection? "},
        "outputs": {"answer": "ReAct integrates reasoning and acting, performing actions - such tools like Wikipedia search API - and then observing / reasoning about the tool outputs."},
    },
    {
        "inputs": {"question": "What are the types of biases that can arise with few-shot prompting?"},
        "outputs": {"answer": "The biases that can arise with few-shot prompting include (1) Majority label bias, (2) Recency bias, and (3) Common token bias."},
    },
    {
        "inputs": {"question": "What are five types of adversarial attacks?"},
        "outputs": {"answer": "Five types of adversarial attacks are (1) Token manipulation, (2) Gradient based attack, (3) Jailbreak prompting, (4) Human red-teaming, (5) Model red-teaming."},
    }
]

### create the daatset and example in LAngsmith
dataset_name="RAG Test Evaluation"
dataset = client.create_dataset(dataset_name=dataset_name)
client.create_examples(
    dataset_id=dataset.id,
    examples=examples
)



{'example_ids': ['23154355-4a6e-4954-8feb-8a49b35795b2',
  '2d1b9a82-8f33-4ad3-910f-074eb17da56e',
  '68c45960-966c-4cc2-a456-844661d4e0f5'],
 'count': 3,
 'as_of': '2026-07-02T06:40:11.863080669Z'}

### Evaluators or Metrics
1. Correctness: Response vs reference answer
- Goal: Measure "how similar/correct is the RAG chain answer, relative to a ground-truth answer"
- Mode: Requires a ground truth (reference) answer supplied through a dataset
- Evaluator: Use LLM-as-judge to assess answer correctness.

In [23]:
from typing_extensions import Annotated, TypedDict
from langchain_openai import ChatOpenAI
import os


# Output schema
class CorrectnessGrade(TypedDict):
    explanation: Annotated[
        str,
        ...,
        "Explain your reasoning for the score"
    ]
    correct: Annotated[
        bool,
        ...,
        "True if the answer is correct, False otherwise."
    ]


correctness_instructions = """You are a teacher grading a quiz.

You will be given a QUESTION, the GROUND TRUTH (correct) ANSWER,
and the STUDENT ANSWER.

Here is the grade criteria to follow:
(1) Grade the student answers based ONLY on their factual accuracy relative to the ground truth answer.
(2) Ensure that the student answer does not contain any conflicting statements.
(3) It is OK if the student answer contains more information than the ground truth answer, as long as it is factually accurate.

Correctness:
A correctness value of True means that the student's answer meets all of the criteria.
A correctness value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning step by step.
"""


grader_llm = ChatOpenAI(
    model="meta-llama/Llama-3.1-8B-Instruct",
    base_url="https://router.huggingface.co/v1",
    api_key=os.environ["HUGGINGFACEHUB_API_TOKEN"],
    temperature=0,
).with_structured_output(CorrectnessGrade)


def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    answers = f"""
QUESTION: {inputs['question']}
GROUND TRUTH ANSWER: {reference_outputs['answer']}
STUDENT ANSWER: {outputs['response']}
"""

    grade = grader_llm.invoke([
        {"role": "system", "content": correctness_instructions},
        {"role": "user", "content": answers},
    ])

    return grade["correct"]

### Relevance: Response vs input
The flow is similar to above, but we simply look at the inputs and outputs without needing the reference_outputs. Without a reference answer we can't grade accuracy, but can still grade relevance—as in, did the model address the user's question or not.

In [24]:
from typing_extensions import Annotated, TypedDict
from langchain_openai import ChatOpenAI
import os


# Output schema
class RelevanceGrade(TypedDict):
    explanation: Annotated[
        str,
        ...,
        "Explain your reasoning for the score"
    ]
    relevant: Annotated[
        bool,
        ...,
        "Provide the score on whether the answer addresses the question"
    ]


# Prompt
relevance_instructions = """You are a teacher grading a quiz.

You will be given a QUESTION and a STUDENT ANSWER.

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is concise and relevant to the QUESTION.
(2) Ensure the STUDENT ANSWER helps to answer the QUESTION.

Relevance:
A relevance value of True means that the student's answer meets all of the criteria.
A relevance value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning step by step.
"""


# Llama grader
relevance_llm = ChatOpenAI(
    model="meta-llama/Llama-3.1-8B-Instruct",
    base_url="https://router.huggingface.co/v1",
    api_key=os.environ["HUGGINGFACEHUB_API_TOKEN"],
    temperature=0,
).with_structured_output(RelevanceGrade)


# Evaluator
def relevance(inputs: dict, outputs: dict) -> bool:
    answer = f"""
QUESTION: {inputs['question']}
STUDENT ANSWER: {outputs['response']}
"""

    grade = relevance_llm.invoke([
        {"role": "system", "content": relevance_instructions},
        {"role": "user", "content": answer}
    ])

    return grade["relevant"]

In [27]:
from langchain_openai import ChatOpenAI
import os

base_llm = ChatOpenAI(
    model="meta-llama/Llama-3.1-8B-Instruct",
    base_url="https://router.huggingface.co/v1",
    api_key=os.environ["HUGGINGFACEHUB_API_TOKEN"],
    temperature=0,
)

### Groundedness: Response vs retrieved docs
Another useful way to evaluate responses without needing reference answers is to check if the response is justified by (or "grounded in") the retrieved documents.

In [28]:
from typing_extensions import Annotated, TypedDict


class GroundedGrade(TypedDict):
    explanation: Annotated[
        str,
        ...,
        "Explain your reasoning for the score"
    ]
    grounded: Annotated[
        bool,
        ...,
        "Provide the score on if the answer hallucinates from the documents"
    ]


grounded_instructions = """You are a teacher grading a quiz.

You will be given FACTS and a STUDENT ANSWER.

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is grounded in the FACTS.
(2) Ensure the STUDENT ANSWER does not contain hallucinated information outside the scope of the FACTS.

Grounded:
A grounded value of True means that the student's answer meets all of the criteria.
A grounded value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning step by step.
"""


grounded_llm = base_llm.with_structured_output(GroundedGrade)


def groundedness(inputs: dict, outputs: dict) -> bool:
    doc_string = "\n\n".join(
        doc.page_content for doc in outputs["documents"]
    )

    answer = f"""
FACTS:
{doc_string}

STUDENT ANSWER:
{outputs['response']}
"""

    grade = grounded_llm.invoke([
        {"role": "system", "content": grounded_instructions},
        {"role": "user", "content": answer},
    ])

    return grade["grounded"]

### Retrieval Relevance: Retrieved docs vs input

In [29]:
from typing_extensions import Annotated, TypedDict


# Output schema
class RetrievalRelevanceGrade(TypedDict):
    explanation: Annotated[
        str,
        ...,
        "Explain your reasoning for the score"
    ]
    relevant: Annotated[
        bool,
        ...,
        "True if the retrieved documents are relevant to the question, False otherwise"
    ]


# Prompt
retrieval_relevance_instructions = """You are a teacher grading a quiz.

You will be given a QUESTION and a set of FACTS provided by the student.

Here is the grade criteria to follow:
(1) Your goal is to identify FACTS that are completely unrelated to the QUESTION.
(2) If the facts contain ANY keywords or semantic meaning related to the question, consider them relevant.
(3) It is OK if the facts have SOME unrelated information as long as (2) is met.

Relevance:
A relevance value of True means that the FACTS contain ANY keywords or semantic meaning related to the QUESTION.
A relevance value of False means that the FACTS are completely unrelated to the QUESTION.

Explain your reasoning step by step.
"""


# Llama grader
retrieval_relevance_llm = base_llm.with_structured_output(
    RetrievalRelevanceGrade
)


def retrieval_relevance(inputs: dict, outputs: dict) -> bool:
    """Evaluator for document relevance."""

    doc_string = "\n\n".join(
        doc.page_content for doc in outputs["documents"]
    )

    prompt = f"""
FACTS:
{doc_string}

QUESTION:
{inputs['question']}
"""

    grade = retrieval_relevance_llm.invoke([
        {"role": "system", "content": retrieval_relevance_instructions},
        {"role": "user", "content": prompt},
    ])

    return grade["relevant"]

### Run the evaluation

In [31]:
def retrieval_relevance(inputs, outputs):
    doc_string = "\n\n".join(
        doc.page_content for doc in outputs["documents"]
    )

    prompt = f"""
QUESTION:
{inputs['question']}

FACTS:
{doc_string}

Reply with ONLY one word:

TRUE -> facts are relevant
FALSE -> facts are unrelated
"""

    result = base_llm.invoke(prompt).content.strip().upper()

    return "TRUE" in result

In [32]:
print(rag_bot("What is prompt engineering?"))

{'answer': "Prompt engineering refers to methods for communicating with language models to steer their behavior for desired outcomes without updating the model weights. It's an empirical science that aims to achieve alignment and model steerability through experimentation and heuristics.", 'documents': [Document(id='5a5fe130-7591-468f-94c4-233316f97117', metadata={'source': 'https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/', 'title': "Prompt Engineering | Lil'Log", 'description': 'Prompt Engineering, also known as In-Context Prompting, refers to methods for how to communicate with LLM to steer its behavior for desired outcomes without updating the model weights. It is an empirical science and the effect of prompt engineering methods can vary a lot among models, thus requiring heavy experimentation and heuristics.\nThis post only focuses on prompt engineering for autoregressive language models, so nothing with Cloze tests, image generation or multimodality models. At it